In [2]:

import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import cv2
import numpy as np
from ultralytics import YOLO
from collections import defaultdict
import time
import os

# =========================
# CONFIG
# =========================
VIDEO_PATH = "clip.mp4"
OUTPUT_PATH = "output.mp4"

MAX_DISTANCE = 1000
MAX_SPEED = 50
GROUND_RATIO = 0.65

IMPORTANT_CLASSES = {"person", "car", "bicycle", "motorcycle"}

priority_map = {
    "person": 1.0,
    "car": 0.9,
    "bicycle": 0.8,
    "motorcycle": 0.85,
    "dog": 0.6,
}

# =========================
# CHECK FILES
# =========================
if not os.path.exists(VIDEO_PATH):
    raise FileNotFoundError(f"Video file not found: {VIDEO_PATH}")

# =========================
# MODEL
# =========================
model = YOLO("yolo11n.pt")
track_history = defaultdict(list)

# =========================
# FUNCTIONS
# =========================
def estimate_distance(box):
    x1, y1, x2, y2 = box
    area = max((x2 - x1) * (y2 - y1), 1)
    return MAX_DISTANCE / area

def normalize(value, max_val, inverse=False):
    val = min(value / max_val, 1.0)
    return 1 - val if inverse else val

def compute_ttc(distance, speed):
    if speed < 1:
        return float("inf")
    return distance / speed

def compute_risk(obj_class, distance, speed, position_score):
    object_priority = priority_map.get(obj_class, 0.1)

    distance_norm = normalize(distance, MAX_DISTANCE, inverse=True)
    speed_norm = normalize(speed, MAX_SPEED)

    if distance_norm < 0.15:
        return 0.05

    score = (
        3 * object_priority +
        6 * distance_norm +
        2 * speed_norm +
        2 * position_score
    )

    return score / 13

def risk_level(score):
    if score > 0.75:
        return "HIGH"
    elif score > 0.4:
        return "MEDIUM"
    else:
        return "LOW"

def predict_collision(track, center, frame_width):
    if len(track) < 5:
        return False

    PATH_WIDTH_RATIO = 0.2
    PATH_HALF_WIDTH = int((PATH_WIDTH_RATIO * frame_width) / 2)

    pts = np.array(track[-5:], dtype=np.float32)
    current = pts[-1]
    velocity = pts[-1] - pts[0]

    if abs(current[0] - center[0]) > PATH_HALF_WIDTH:
        return False

    if velocity[1] < 2:
        return False

    direction_to_center = center - current
    if np.dot(velocity, direction_to_center) <= 0:
        return False

    speed = np.linalg.norm(velocity)
    if speed < 5:
        return False

    future = current + velocity * 1.2

    if abs(future[0] - center[0]) > PATH_HALF_WIDTH:
        return False

    return True

def avoidance_action(level, collision, cx, center_x):
    if not collision:
        return "NONE"

    if level == "HIGH":
        return "STOP"

    return "LEFT" if cx > center_x else "RIGHT"



cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise RuntimeError("Cannot open video source")

FRAME_WIDTH = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
FRAME_HEIGHT = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
FPS = cap.get(cv2.CAP_PROP_FPS) or 30

CENTER = np.array([FRAME_WIDTH // 2, FRAME_HEIGHT // 2], dtype=np.float32)
GROUND_THRESHOLD = int(GROUND_RATIO * FRAME_HEIGHT)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(OUTPUT_PATH, fourcc, FPS, (FRAME_WIDTH, FRAME_HEIGHT))

if not out.isOpened():
    raise RuntimeError("Cannot create video writer")


# =========================
# MAIN LOOP
# =========================
prev_time = time.time()
frame_count = 0

while True:
    ret, frame = cap.read()

    if not ret:
        break

    current_time = time.time()
    dt = max(current_time - prev_time, 1e-3)
    prev_time = current_time

    try:
        results = model.track(
            frame,
            persist=True,
            device="cpu",
            verbose=False
        )
    except Exception as e:
        print("Tracking error:", e)
        break

    if results and len(results) > 0:

        boxes_obj = results[0].boxes

        if boxes_obj is not None and boxes_obj.id is not None:

            boxes = boxes_obj.xyxy.cpu().numpy()
            ids = boxes_obj.id.cpu().numpy().astype(int)
            classes = boxes_obj.cls.cpu().numpy().astype(int)

            for box, track_id, cls in zip(boxes, ids, classes):

                obj_name = model.names[int(cls)]

                if obj_name not in IMPORTANT_CLASSES:
                    continue

                x1, y1, x2, y2 = map(int, box)

                foot_x = (x1 + x2) // 2
                foot_y = y2

                track = track_history[track_id]
                track.append((foot_x, foot_y))

                if len(track) > 20:
                    track.pop(0)

                distance = estimate_distance(box)

                speed = 0
                if len(track) > 1:
                    dx = track[-1][0] - track[-2][0]
                    dy = track[-1][1] - track[-2][1]
                    speed = np.sqrt(dx**2 + dy**2) / dt

                pos_dist = np.linalg.norm(
                    np.array([foot_x, foot_y], dtype=np.float32) - CENTER
                )

                position_score = 1 - min(
                    pos_dist / np.linalg.norm(CENTER),
                    1
                )

                score = compute_risk(
                    obj_name,
                    distance,
                    speed,
                    position_score
                )

                level = risk_level(score)

                if foot_y < GROUND_THRESHOLD:
                    collision = False
                else:
                    collision = predict_collision(
                        track,
                        CENTER,
                        FRAME_WIDTH
                    )

                ttc = compute_ttc(distance, speed)

                if ttc < 1.5 and speed > 5 and distance < 300:
                    collision = True

                if len(track) < 3:
                    collision = False

                action = avoidance_action(
                    level,
                    collision,
                    foot_x,
                    CENTER[0]
                )

                color = (0, 255, 0)

                if level == "MEDIUM":
                    color = (0, 255, 255)

                elif level == "HIGH":
                    color = (0, 0, 255)

                label = f"{obj_name} | {level} | {score:.2f} | {action}"

                if collision:
                    label += " | COLLISION!"

                cv2.rectangle(
                    frame,
                    (x1, y1),
                    (x2, y2),
                    color,
                    2
                )

                cv2.putText(
                    frame,
                    label,
                    (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.5,
                    color,
                    2
                )

                for i in range(1, len(track)):
                    cv2.line(
                        frame,
                        tuple(map(int, track[i-1])),
                        tuple(map(int, track[i])),
                        (255, 0, 0),
                        2
                    )

    cv2.imshow("Risk-Aware Navigation", frame)
    out.write(frame)
    frame_count += 1
    
    if cv2.waitKey(1) & 0xFF == 27:  # ESC to exit
        break

cap.release()
out.release()
cv2.destroyAllWindows()

print("Done. Output saved:", OUTPUT_PATH)

KeyboardInterrupt: 

In [ ]:

import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import cv2 
import numpy as np
from ultralytics import YOLO
from collections import defaultdict
import time


# CONFIG

USE_OUTPUT = True
OUTPUT_PATH = "output.mp4"

MAX_DISTANCE = 1000
MAX_SPEED = 50
GROUND_RATIO = 0.65

IMPORTANT_CLASSES = {"person", "car", "bicycle", "motorcycle"}

priority_map = {
    "person": 1.0,
    "car": 0.9,
    "bicycle": 0.8,
    "motorcycle": 0.85,
    "dog": 0.6,
}


# MODEL

model = YOLO("yolo11n.pt")
track_history = defaultdict(list)


# FUNCTIONS

def estimate_distance(box):
    x1, y1, x2, y2 = box
    area = max((x2 - x1) * (y2 - y1), 1)
    return MAX_DISTANCE / area

def normalize(value, max_val, inverse=False):
    val = min(value / max_val, 1.0)
    return 1 - val if inverse else val

def compute_ttc(distance, speed):
    return float("inf") if speed < 1 else distance / speed

def compute_risk(obj_class, distance, speed, position_score):
    object_priority = priority_map.get(obj_class, 0.1)

    distance_norm = normalize(distance, MAX_DISTANCE, inverse=True)
    speed_norm = normalize(speed, MAX_SPEED)

    if distance_norm < 0.15:
        return 0.05

    score = (
        3 * object_priority +
        6 * distance_norm +
        2 * speed_norm +
        2 * position_score
    )
    return score / 13

def risk_level(score):
    if score > 0.75:
        return "HIGH"
    elif score > 0.4:
        return "MEDIUM"
    return "LOW"

def predict_collision(track, center, frame_width):
    if len(track) < 5:
        return False

    PATH_HALF_WIDTH = int((0.2 * frame_width) / 2)

    pts = np.array(track[-5:], dtype=np.float32)
    current = pts[-1]
    velocity = pts[-1] - pts[0]

    if abs(current[0] - center[0]) > PATH_HALF_WIDTH:
        return False
    if velocity[1] < 2:
        return False
    if np.dot(velocity, center - current) <= 0:
        return False
    if np.linalg.norm(velocity) < 5:
        return False

    future = current + velocity * 1.2
    return abs(future[0] - center[0]) <= PATH_HALF_WIDTH

def avoidance_action(level, collision, cx, center_x):
    if not collision:
        return "NONE"
    if level == "HIGH":
        return "STOP"
    return "LEFT" if cx > center_x else "RIGHT"


cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError("Cannot access webcam")

FRAME_WIDTH = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
FRAME_HEIGHT = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
FPS = cap.get(cv2.CAP_PROP_FPS) or 30

CENTER = np.array([FRAME_WIDTH // 2, FRAME_HEIGHT // 2], dtype=np.float32)
GROUND_THRESHOLD = int(GROUND_RATIO * FRAME_HEIGHT)


if USE_OUTPUT:
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(OUTPUT_PATH, fourcc, FPS, (FRAME_WIDTH, FRAME_HEIGHT))


prev_time = time.time()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    current_time = time.time()
    dt = max(current_time - prev_time, 1e-3)
    prev_time = current_time

    results = model.track(frame, persist=True, device="cpu", verbose=False)

    if results and results[0].boxes is not None and results[0].boxes.id is not None:

        boxes = results[0].boxes.xyxy.cpu().numpy()
        ids = results[0].boxes.id.cpu().numpy().astype(int)
        classes = results[0].boxes.cls.cpu().numpy().astype(int)

        for box, track_id, cls in zip(boxes, ids, classes):

            obj_name = model.names[int(cls)]
            if obj_name not in IMPORTANT_CLASSES:
                continue

            x1, y1, x2, y2 = map(int, box)

            foot_x = (x1 + x2) // 2
            foot_y = y2

            track = track_history[track_id]
            track.append((foot_x, foot_y))
            if len(track) > 20:
                track.pop(0)

            distance = estimate_distance(box)

            speed = 0
            if len(track) > 1:
                dx = track[-1][0] - track[-2][0]
                dy = track[-1][1] - track[-2][1]
                speed = np.sqrt(dx**2 + dy**2) / dt

            pos_dist = np.linalg.norm(
                np.array([foot_x, foot_y], dtype=np.float32) - CENTER
            )
            position_score = 1 - min(pos_dist / np.linalg.norm(CENTER), 1)

            score = compute_risk(obj_name, distance, speed, position_score)
            level = risk_level(score)

            collision = False
            if foot_y >= GROUND_THRESHOLD:
                collision = predict_collision(track, CENTER, FRAME_WIDTH)

            ttc = compute_ttc(distance, speed)
            if ttc < 1.5 and speed > 5 and distance < 300:
                collision = True

            if len(track) < 3:
                collision = False

            action = avoidance_action(level, collision, foot_x, CENTER[0])

            # Visualization
            color = (0, 255, 0)
            if level == "MEDIUM":
                color = (0, 255, 255)
            elif level == "HIGH":
                color = (0, 0, 255)

            label = f"{obj_name} | {level} | {score:.2f} | {action}"
            if collision:
                label += " | COLLISION!"

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, label, (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    cv2.imshow("Risk-Aware Navigation (Webcam)", frame)

    if USE_OUTPUT:
        out.write(frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
if USE_OUTPUT:
    out.release()
cv2.destroyAllWindows()

print("Done.")

Error: File does not exist -> your_image.jpg
Cannot display 'Original Image' because image is None.
Cannot convert to grayscale because image is None.
Cannot display 'Grayscale Image' because image is None.
Cannot resize because image is None.
Cannot display 'Resized Image' because image is None.
Cannot apply filters because image is None.
Cannot display 'Average Blur' because image is None.
Cannot display 'Gaussian Blur' because image is None.
Cannot display 'Median Blur' because image is None.
Cannot detect edges because grayscale image is None.
Cannot display 'Canny Edge Detection' because image is None.
Garbage collection completed.
